# Pipeline de Datos — Superstore

Flujo completo de 6 pasos sobre `train.csv` (Superstore dataset):
`LOAD → INSPECT → CLEAN → TRANSFORM → ANALYZE → EXPORT`

Cada paso imprime una línea de log para saber qué ocurrió y con cuántos datos.

In [2]:
import pandas as pd
from pathlib import Path

# Rutas
FILEPATH = Path('../../data/external/train.csv')
OUTPUTS  = Path('../../data/processed')
OUTPUTS.mkdir(parents=True, exist_ok=True)

## 1 - LOAD

Se carga el CSV y se guarda una copia sin modificar (`df_original`).
El log confirma cuántas filas entraron al pipeline.

In [ ]:
# El def es importante si es necesario usar el bloque de código en otro lugar, 
# permite ser llamada con distintos parámetros sin repetir el código
def step_load(filepath):
    df = pd.read_csv(filepath)
    df_original = df.copy()
    print(f'[LOAD]      {df.shape[0]:,} filas | {df.shape[1]} columnas')
    return df, df_original

## 2 - INSPECT

Se detectan nulos y duplicados sin modificar nada todavía.
Este paso sirve de diagnóstico: determina qué hay que limpiar en el paso siguiente.

In [1]:
def step_inspect(df):
    nulos      = df.isnull().sum().sum()# se prefiere el conteo sobre todo el dataset, en caso de utiliza solamente un 
                                        #.sum() se mostraría los nulos por columna y no totales
    duplicados = df.duplicated().sum()
    print(f'[INSPECT]   {nulos} nulos totales | {duplicados} filas duplicadas')
    if nulos > 0:
        detalle = df.isnull().sum()
        print(detalle[detalle > 0].to_string())
    return nulos, duplicados

## 3 - CLEAN

Pasos de limpieza aplicados sobre el dataset Superstore:

- `Order Date` viene como string `DD/MM/YYYY` → se convierte a datetime
- Se eliminan filas duplicadas
- `Customer Name` nulo → se rellena con `'Desconocido'`
- `Sales` nulo → se elimina la fila (sin importe no hay análisis posible)
- `City` → se normaliza con `.strip().str.title()` para evitar inconsistencias

In [5]:
def step_clean(df):
    df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
    df = df.drop_duplicates()
    df['Customer Name'] = df['Customer Name'].fillna('Desconocido')
    df = df.dropna(subset=['Sales'])
    df['City'] = df['City'].str.strip().str.title()
    print(f'[CLEAN]     {df.shape[0]:,} filas limpias')
    return df

## 4 - TRANSFORM

Se derivan columnas nuevas a partir de las existentes:

- `mes` y `trimestre` desde `Order Date`
- `revenue_cliente`: revenue acumulado por `Customer ID` usando `transform('sum')`.
  `transform` devuelve una columna del mismo tamaño que el DataFrame original,
  por eso cada fila muestra el total de su cliente, no un resumen colapsado.
- `margen_categoria`: para comparar cada venta contra el revenue total de su categoría
  (útil para detectar si una venta aporta más o menos que la media del grupo)

In [6]:
def step_transform(df):
    df = df.copy()
    df['mes']       = df['Order Date'].dt.month
    df['trimestre'] = df['Order Date'].dt.quarter
    df['anio']      = df['Order Date'].dt.year

    # Revenue acumulado por cliente (transform mantiene el shape original)
    df['revenue_cliente'] = (
        df.groupby('Customer ID')['Sales'].transform('sum').round(2)
    )

    # Porcentaje que representa cada venta sobre el total de su categoría
    total_categoria = df.groupby('Category')['Sales'].transform('sum')
    df['pct_categoria'] = (df['Sales'] / total_categoria * 100).round(2)

    print(f'[TRANSFORM] {df.shape[1]} columnas tras transformación')
    return df

## 5 - ANALYZE

Se agrupan las ventas por `Category` con tres métricas:

- `num_ventas`: cuántas líneas de venta hay (COUNT)
- `revenue_total`: suma de ventas del grupo (SUM)
- `ticket_medio`: venta promedio por transacción (AVG)

El resultado se ordena de mayor a menor revenue y se redondea a 2 decimales.

In [7]:
def step_analyze(df):
    resultado = (
        df.groupby('Category')
        .agg(
            num_ventas    = ('Sales', 'count'),
            revenue_total = ('Sales', 'sum'),
            ticket_medio  = ('Sales', 'mean')
        )
        .reset_index()
        .sort_values('revenue_total', ascending=False)
        .round(2)
    )
    print(f'[ANALYZE]   {len(resultado)} categorías analizadas')
    return resultado

## 6 - EXPORT

Se guarda el resultado en `data/processed/resultado_pipeline.csv`.
La carpeta `processed` ya existe (se creó en la celda de imports).

In [8]:
def step_export(resultado, outputs_path):
    out = outputs_path / 'resultado_pipeline.csv'
    resultado.to_csv(out, index=False)
    print(f'[EXPORT]    guardado en {out}')

## Ejecutar el pipeline completo

Se llama a cada paso en orden y se encadena la salida de uno como entrada del siguiente.

In [9]:
def ejecutar_pipeline(filepath):
    print('=' * 50)
    print('INICIANDO PIPELINE')
    print('=' * 50)

    df, df_original = step_load(filepath)
    step_inspect(df)
    df = step_clean(df)
    df = step_transform(df)
    resultado = step_analyze(df)
    step_export(resultado, OUTPUTS)

    print('=' * 50)
    print('PIPELINE COMPLETADO')
    print('=' * 50)

    return df, resultado


df_clean, resultado = ejecutar_pipeline(FILEPATH)
print()
print(resultado)

INICIANDO PIPELINE
[LOAD]      9,800 filas | 18 columnas
[INSPECT]   11 nulos totales | 0 filas duplicadas
Postal Code    11
[CLEAN]     9,800 filas limpias
[TRANSFORM] 23 columnas tras transformación
[ANALYZE]   3 categorías analizadas
[EXPORT]    guardado en ..\..\data\processed\resultado_pipeline.csv
PIPELINE COMPLETADO

          Category  num_ventas  revenue_total  ticket_medio
2       Technology        1813      827455.87        456.40
0        Furniture        2078      728658.58        350.65
1  Office Supplies        5909      705422.33        119.38
